In [ ]:
from transformers import AutoModelForCausalLM, AutoTokenizer
import torch

model_name = "microsoft/DialoGPT-small"
tokenizer = AutoTokenizer.from_pretrained(model_name)
model = AutoModelForCausalLM.from_pretrained(model_name)

knowledge_base = {
    "what is artificial intelligence": "Artificial Intelligence is the simulation of human intelligence in machines that can learn, reason, and solve problems.",
    "what is ai": "Artificial Intelligence is the simulation of human intelligence in machines that can learn, reason, and solve problems.",
    "who created python": "Python was created by Guido van Rossum and released in 1991.",
    "who invented python": "Python was created by Guido van Rossum and released in 1991.",
}

def chatbot():
    print("Chatbot: Hello! I am your AI assistant. Ask me anything. Type 'exit' to quit.\n")

    chat_history_ids = None

    while True:
        user_input = input("You: ").lower()

        if user_input in ["exit", "quit"]:
            print("Chatbot: Goodbye! Have a great day 😊")
            break

        if user_input in knowledge_base:
            print("Chatbot:", knowledge_base[user_input])
            continue

        new_input_ids = tokenizer.encode(user_input + tokenizer.eos_token, return_tensors='pt')

        if chat_history_ids is not None:
            bot_input_ids = torch.cat([chat_history_ids, new_input_ids], dim=-1)
        else:
            bot_input_ids = new_input_ids

        attention_mask = torch.ones(bot_input_ids.shape, dtype=torch.long)

        chat_history_ids = model.generate(
            bot_input_ids,
            attention_mask=attention_mask,
            max_length=200,
            pad_token_id=tokenizer.eos_token_id,
            do_sample=True,
            top_k=40,
            top_p=0.9,
            temperature=0.7
        )

        response = tokenizer.decode(
            chat_history_ids[:, bot_input_ids.shape[-1]:][0],
            skip_special_tokens=True
        )

        print("Chatbot:", response.strip())

chatbot()